# Step 1: Environment Setup

In this step, we:
- Install required libraries
- Import key packages
- Prepare our Colab notebook for data parsing and future API interaction

Note: We are **not** loading the OpenAI API yet — that happens later in Phase 2.


In [1]:
# Install required packages
!pip install openai python-dotenv pandas scikit-learn --quiet

# Import standard libraries
import pandas as pd
import numpy as np
import re
import os

# For ML model later
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


# Step 2: Load and Inspect Raw Data

In this step, we:
- Upload our raw CSV to Colab
- Load it using pandas
- Inspect a few sample rows to confirm the structure

We expect three columns: `headline`, `id`, and `updatedDate`.


In [2]:
# Upload your raw CSV
from google.colab import files
uploaded = files.upload()

# Load CSV into DataFrame
file_name = list(uploaded.keys())[0]  # Get uploaded filename
df = pd.read_csv(file_name)

# Quick structure check
print("Number of rows:", len(df))
df.head(5)


Saving heards_full_dataset.csv to heards_full_dataset.csv
Number of rows: 215400


,headline,id,updatedDate
0,"Platts HSFO Med Crg CIF bss Malta 10-25, GLTD ...",c7c0b94e-3a22-4f47-88c4-db05f07d1073,2025-05-23T14:44:56.674Z
1,Platts Singapore Fuel Oil Bids Offers Trades,194ab12c-94d5-49de-a881-c296875c91c8,2025-05-23T11:03:11Z
2,PLATTS HSFO FOB FUJ: TRADE SUMMARY: No trade,2a5943d5-8e18-44ed-b059-17985e9040a9,2025-05-23T09:33:44.230Z
3,PLATTS HSFO FOB FUJ: PHYSICAL OFFERS FINALS ON...,7e6d7ef7-7449-434e-b5c2-54cb94750ec1,2025-05-23T09:33:42.718Z
4,PLATTS HSFO FOB FUJ: PHYSICAL BIDS FINALS ON C...,23dc30dc-7474-431e-a08a-2258a333e1b6,2025-05-23T09:33:38.922Z


# Step 3: Create a Safe Copy of the Raw Data

Before applying any transformation, we make a full copy of our original dataset.
This protects the raw data and allows us to:
- Compare raw vs. structured headlines
- Re-run failed parses or experiments
- Export only parsed rows if needed


In [3]:
# Make a clean copy of the uploaded raw data
df_parsed = df.copy()

print(f"✅ Data copied. Total rows: {len(df_parsed)}")
df_parsed.head()


✅ Data copied. Total rows: 215400


,headline,id,updatedDate
0,"Platts HSFO Med Crg CIF bss Malta 10-25, GLTD ...",c7c0b94e-3a22-4f47-88c4-db05f07d1073,2025-05-23T14:44:56.674Z
1,Platts Singapore Fuel Oil Bids Offers Trades,194ab12c-94d5-49de-a881-c296875c91c8,2025-05-23T11:03:11Z
2,PLATTS HSFO FOB FUJ: TRADE SUMMARY: No trade,2a5943d5-8e18-44ed-b059-17985e9040a9,2025-05-23T09:33:44.230Z
3,PLATTS HSFO FOB FUJ: PHYSICAL OFFERS FINALS ON...,7e6d7ef7-7449-434e-b5c2-54cb94750ec1,2025-05-23T09:33:42.718Z
4,PLATTS HSFO FOB FUJ: PHYSICAL BIDS FINALS ON C...,23dc30dc-7474-431e-a08a-2258a333e1b6,2025-05-23T09:33:38.922Z


# Step 4: Apply Regex-Based Parsing to Extract Structured Columns

Now we apply the `parse_headline()` function (defined earlier) to the `headline` column.
This function uses regular expressions to extract structured information:
- commodity, grade, incoterm, location
- action_type, party, price, price_basis
- start_date, end_date, volume_min, volume_max, frequency_tag

The structured results will be stored in new columns in `df_parsed`.


In [4]:
def parse_headline(text):
    result = {
        "commodity": None,
        "grade": None,
        "incoterm": None,
        "location": None,
        "action_type": None,
        "party": None,
        "price": None,
        "price_basis": None,
        "start_date": None,
        "end_date": None,
        "volume_min": None,
        "volume_max": None,
        "frequency_tag": None,
        "source": "regex"
    }

    text = str(text)
    text_lower = text.lower()

    # Month mapping for datetime formatting
    month_map = {
        "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
        "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
        "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
    }

    # 1. Commodity & Grade
    if "hsfo" in text_lower:
        result["commodity"] = "HSFO"
    grade_match = re.search(r'(180|380)\s*cst', text, re.IGNORECASE)
    if grade_match:
        result["grade"] = int(grade_match.group(1))

    # 2. Incoterm
    for term in ["FOB", "CIF", "DAP", "EXW", "FCA", "DDP", "CFR", "CPT", "DES"]:
        if term in text:
            result["incoterm"] = term
            break

    # 3. Location
    loc_match = re.search(r'\b(Straits|Fujairah|Fuj|FUJ|Malta|Asia|Singapore)\b', text, re.IGNORECASE)
    if loc_match:
        loc = loc_match.group(1).capitalize()
        result["location"] = "Fujairah" if loc.lower().startswith("fuj") else loc
        result["location"] = result["location"].title()

    # 4. Action Type (multi-variant match)
    if re.search(r"no\s+(physical\s+)?trade(s)?|not traded|no transaction", text_lower):
        result["action_type"] = "No Trade"
    elif re.search(r"no\s+(physical\s+)?bid(s)?( seen| heard| posted| available)?", text_lower):
        result["action_type"] = "No Bid"
    elif re.search(r"no\s+(physical\s+)?offer(s)?( seen| heard| posted| available)?", text_lower):
        result["action_type"] = "No Offer"
    elif re.search(r"trade summary|executed|traded", text_lower):
        result["action_type"] = "Trade Summary"
    elif re.search(r"(raises|hikes|increases|lifts)\s+(bid|offer)", text_lower):
        result["action_type"] = "Raise"
    elif re.search(r"(lowers|cuts|drops|decreases)\s+(bid|offer)", text_lower):
        result["action_type"] = "Lower"
    elif re.search(r"\b(bid|bids|bidding|posted bid)\b", text_lower):
        result["action_type"] = "Bid"
    elif re.search(r"\b(offer|offers|offering|posted offer)\b", text_lower):
        result["action_type"] = "Offer"

    # 5. Party
    party_match = re.search(r',\s*([A-Z0-9]+)\s+(bids|offers|raises|lowers)', text)
    if party_match:
        result["party"] = party_match.group(1).strip(":")

    # 6. Date Range
    date_match = re.search(r'([A-Z][a-z]{2})\s*(\d{1,2})\s*-\s*([A-Z][a-z]{2})?\s*(\d{1,2})', text)
    if date_match:
        sm = date_match.group(1)
        sd = int(date_match.group(2))
        em = date_match.group(3) if date_match.group(3) else sm
        ed = int(date_match.group(4))
        sm_num = month_map.get(sm, "01")
        em_num = month_map.get(em, sm_num)
        result["start_date"] = f"2025-{sm_num}-{sd:02d}"
        result["end_date"] = f"2025-{em_num}-{ed:02d}"

    # 7. Price
    price_match = re.search(r'\$(\-?\d+\.\d+)', text)
    if price_match:
        result["price"] = float(price_match.group(1))

    # 8. Price Basis
    basis_match = re.search(r'100%\s+([A-Z0-9]+(?:\s*[A-Z0-9]+)?)', text)
    if basis_match:
        result["price_basis"] = basis_match.group(1).strip()

    # 9. Volume (split range)
    vol_match = re.search(r'for\s+(\d{1,3}-\d{1,3})', text)
    if vol_match:
        vmin, vmax = vol_match.group(1).split("-")
        result["volume_min"] = int(vmin)
        result["volume_max"] = int(vmax)

    # 10. Frequency Tag
    freq_match = re.search(r'(Any Day|5 Day|BalMnth|Bal Month)', text)
    if freq_match:
        result["frequency_tag"] = freq_match.group(1)

    return result


In [5]:
# Apply the parser to each headline and convert the output to structured columns
parsed_results = df_parsed['headline'].apply(parse_headline).apply(pd.Series)

# Combine parsed fields into df_parsed
df_parsed = pd.concat([df_parsed, parsed_results], axis=1)

# Preview the result
print("✅ Parsing complete. Sample structured output:")
df_parsed.head()


✅ Parsing complete. Sample structured output:


,headline,id,updatedDate,commodity,grade,incoterm,location,action_type,party,price,price_basis,start_date,end_date,volume_min,volume_max,frequency_tag,source
0,"Platts HSFO Med Crg CIF bss Malta 10-25, GLTD ...",c7c0b94e-3a22-4f47-88c4-db05f07d1073,2025-05-23T14:44:56.674Z,HSFO,NaN,CIF,Malta,Offer,GLTD,16.0,3,2025-06-02,2025-06-06,NaN,NaN,Any Day,regex
1,Platts Singapore Fuel Oil Bids Offers Trades,194ab12c-94d5-49de-a881-c296875c91c8,2025-05-23T11:03:11Z,None,NaN,None,Singapore,Bid,None,NaN,None,None,None,NaN,NaN,None,regex
2,PLATTS HSFO FOB FUJ: TRADE SUMMARY: No trade,2a5943d5-8e18-44ed-b059-17985e9040a9,2025-05-23T09:33:44.230Z,HSFO,NaN,FOB,Fujairah,No Trade,None,NaN,None,None,None,NaN,NaN,None,regex
3,PLATTS HSFO FOB FUJ: PHYSICAL OFFERS FINALS ON...,7e6d7ef7-7449-434e-b5c2-54cb94750ec1,2025-05-23T09:33:42.718Z,HSFO,380.0,FOB,Fujairah,No Offer,None,NaN,None,None,None,NaN,NaN,None,regex
4,PLATTS HSFO FOB FUJ: PHYSICAL BIDS FINALS ON C...,23dc30dc-7474-431e-a08a-2258a333e1b6,2025-05-23T09:33:38.922Z,HSFO,380.0,FOB,Fujairah,No Bid,None,NaN,None,None,None,NaN,NaN,None,regex


In [6]:
df_parsed.to_csv("heards_parsed_phase1.csv", index=False)
from google.colab import files
files.download("heards_parsed_phase1.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 4B: Evaluate Field Completion After Regex Parsing

We now check how well our parser filled in key fields.
This will help us identify which rows need GPT or ML completion in Phase 2.


In [7]:
# Check what percentage of each key column is filled
fields_to_check = [
    "action_type", "start_date", "end_date",
    "price_basis", "price", "party",
    "volume_min", "volume_max"
]

# Calculate and display percentage of non-null values per column
completeness = df_parsed[fields_to_check].notnull().mean().sort_values(ascending=False) * 100

print("✅ Regex Parsing Completion (% Filled):")
display(completeness.round(1))


✅ Regex Parsing Completion (% Filled):


,0
action_type,84.1
start_date,82.8
end_date,82.8
price_basis,74.8
price,71.4
party,69.5
volume_min,55.5
volume_max,55.5


# Step 5: GPT Labeling Phase – Setup

We now begin Phase 2 of the HEARDS data cleaning pipeline:
using GPT-4 to extract missing information for rows that regex could not parse.

## What we’re doing:
- Load our `.env` file containing the OpenAI API key
- Configure `openai` to use that key
- Test a small sample of rows (25) that need completion for:
  - `action_type`
  - `price`
  - `party`

We will only label ~1,000–2,000 rows using GPT total. This batch will be used to train a lightweight ML model for the rest.


In [8]:
# Upload .env file
from google.colab import files
uploaded = files.upload()


Saving .env to .env


In [9]:
# Load API key and configure OpenAI client
from dotenv import load_dotenv
import openai
import os

# Load .env file (assumes you uploaded it just now)
load_dotenv(".env")

# Set API key for OpenAI
openai.api_key = os.getenv("OPENAI_API_KEY")

# Verify it's loaded
if openai.api_key:
    print("✅ OpenAI API key loaded successfully.")
else:
    print("❌ Failed to load API key.")


✅ OpenAI API key loaded successfully.


# Step 5A: Extract Sample Rows for GPT Labeling

We'll extract a sample of 25 rows where:
- `action_type` is missing
- `price` is missing
- `party` is missing

These are the rows our regex parser couldn’t handle. GPT will label these fields using the `headline` text.


In [34]:
# Extract rows where all three key fields are missing
gpt_candidates = df_parsed[
    df_parsed['action_type'].isna() &
    df_parsed['price'].isna() &
    df_parsed['party'].isna()
]

# Sample 25 rows for testing
gpt_sample = gpt_candidates[['headline']].sample(25, random_state=42).reset_index(drop=True)

# Preview
print("✅ Sample extracted for GPT labeling:")
gpt_sample.head()


✅ Sample extracted for GPT labeling:


,headline
0,Platts FUJ HSFO 380CST: Apr 29 - May 3 pegged ...
1,Platts HSFO 180CST: Mar 3 - Mar 7 pegged at MO...
2,Asia 113: Platts HSFO: Platts would like to re...
3,Asia 1736: PLATTS SINGAPORE FUEL OIL PAPER TRA...
4,Platts HSFO 180CST: Dec 15 - Dec 19 pegged at ...


# Step 5B: Send Headlines to GPT to Extract Missing Fields

We use GPT-4 to extract:
- `action_type` (e.g., Bid, Offer, No Trade, etc.)
- `price` (if any)
- `party` (company or trader mentioned)

Each row is sent as a prompt, and GPT responds with structured JSON.


In [35]:
from openai import OpenAI
import os, json

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Predefine the reusable system prompt
system_prompt = (
    "You are a financial data extraction assistant.\n"
    "Extract the following fields from the user's message:\n\n"
    "- action_type (choose from: Bid, Offer, Raise, Lower, No Trade, No Bid, No Offer, Trade Summary)\n"
    "- price (float if available, else null)\n"
    "- party (trader name if mentioned, else null)\n\n"
    "Return output as valid JSON with keys: action_type, price, party."
)

def gpt_extract_info_v2(headline):
    try:
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": headline}
            ],
            temperature=0
        )
        content = response.choices[0].message.content.strip()
        print("🧠 GPT Response:\n", content)
        return content
    except Exception as e:
        print(f"❌ Error for headline: {headline[:60]} → {e}")
        return None


In [28]:
import re

gpt_results = []

for i, row in gpt_sample.iterrows():
    raw_output = gpt_extract_info_v1(row['headline'])

    try:
        # Extract the actual JSON content from within markdown formatting
        json_match = re.search(r"\{[\s\S]*\}", raw_output)
        if json_match:
            cleaned = json_match.group(0)
            parsed = json.loads(cleaned)
        else:
            raise ValueError("No valid JSON found in GPT response.")

        gpt_results.append({
            "headline": row['headline'],
            "action_type": parsed.get("action_type"),
            "price": parsed.get("price"),
            "party": parsed.get("party")
        })

    except Exception as e:
        print(f"⚠️ Failed to parse GPT response for row {i} → {e}")
        gpt_results.append({
            "headline": row['headline'],
            "action_type": None,
            "price": None,
            "party": None
        })


🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": 25.25,
  "party": null
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": 298.43,
  "party": null
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": null,
  "party": "Platts"
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "Trade Summary",
  "price": null,
  "party": null
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": 459.59,
  "party": null
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": null,
  "party": null
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": null,
  "party": null
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": null,
  "party": null
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "Bid",
  "price": null,
  "party": "TRAFI"
}
```
🧠 GPT Response:
 ```json
{
  "action_type": "No Trade",
  "price": null,
  "party": "Platts"
}
```
🧠 GPT Response:
 ```json
{
  "acti

In [30]:
# View GPT-labeled results
import pandas as pd
gpt_df = pd.DataFrame(gpt_results)
gpt_df.head()


,headline,action_type,price,party
0,Platts FUJ HSFO 380CST: Apr 29 - May 3 pegged ...,No Trade,25.25,None
1,Platts HSFO 180CST: Mar 3 - Mar 7 pegged at MO...,No Trade,298.43,None
2,Asia 113: Platts HSFO: Platts would like to re...,No Trade,NaN,Platts
3,Asia 1736: PLATTS SINGAPORE FUEL OIL PAPER TRA...,Trade Summary,NaN,None
4,Platts HSFO 180CST: Dec 15 - Dec 19 pegged at ...,No Trade,459.59,None


# Step 5C: Label 2,000 Incomplete Rows with GPT

Now that the sample test worked, we will:
- Extract 2,000 rows that are missing key fields (`action_type`, `price`, `party`)
- Use GPT-4 Turbo to label these fields
- Store the results in a structured DataFrame and export to CSV

We'll batch requests to stay within API limits and budget.


In [37]:
# Select rows missing ALL 3 key fields
gpt_candidates_full = df_parsed[
    df_parsed['action_type'].isna() &
    df_parsed['price'].isna() &
    df_parsed['party'].isna()
].copy()

# Extract up to 2,000 rows
gpt_batch = gpt_candidates_full[['headline']].sample(2000, random_state=42).reset_index(drop=True)

print("✅ Number of rows selected for GPT labeling:", len(gpt_batch))
gpt_batch.head()


✅ Number of rows selected for GPT labeling: 2000


,headline
0,Platts FUJ HSFO 380CST: Apr 29 - May 3 pegged ...
1,Platts HSFO 180CST: Mar 3 - Mar 7 pegged at MO...
2,Asia 113: Platts HSFO: Platts would like to re...
3,Asia 1736: PLATTS SINGAPORE FUEL OIL PAPER TRA...
4,Platts HSFO 180CST: Dec 15 - Dec 19 pegged at ...


In [38]:
import os
import json
import re
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from google.colab import files

# ✅ Load .env and initialize OpenAI client
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ✅ Input batch
# Ensure this exists before running:
# gpt_batch = df_parsed[df_parsed["source"] == "regex"].sample(n=2000, random_state=42)

# ✅ Checkpoint file setup
checkpoint_file = "gpt_labeled_checkpoint.csv"
if os.path.exists(checkpoint_file):
    gpt_labeled_df = pd.read_csv(checkpoint_file)
    start_index = len(gpt_labeled_df)
    print(f"🔁 Resuming from row {start_index}")
else:
    gpt_labeled_df = pd.DataFrame(columns=["headline", "action_type", "price", "party"])
    start_index = 0

# ✅ System instruction
system_prompt = (
    "You are a financial data extraction assistant.\n"
    "Extract the following fields from the user's message:\n\n"
    "- action_type (choose from: Bid, Offer, Raise, Lower, No Trade, No Bid, No Offer, Trade Summary)\n"
    "- price (float if available, else null)\n"
    "- party (trader name if mentioned, else null)\n\n"
    "Return output as valid JSON with keys: action_type, price, party."
)

# ✅ GPT request function
def gpt_extract_info_v2(headline):
    try:
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": headline}
            ],
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ Error for headline: {headline[:60]} → {e}")
        return None

# ✅ Main loop
gpt_labeled = []

for i, row in gpt_batch.iloc[start_index:].iterrows():
    raw_output = gpt_extract_info_v2(row['headline'])

    try:
        json_match = re.search(r"\{[\s\S]*\}", raw_output)
        if json_match:
            cleaned = json_match.group(0)
            parsed = json.loads(cleaned)
        else:
            raise ValueError("No valid JSON found.")

        entry = {
            "headline": row['headline'],
            "action_type": parsed.get("action_type"),
            "price": parsed.get("price"),
            "party": parsed.get("party")
        }
    except Exception as e:
        print(f"⚠️ Row {i} failed → {e}")
        entry = {
            "headline": row['headline'],
            "action_type": None,
            "price": None,
            "party": None
        }

    gpt_labeled.append(entry)

    # ✅ Save every 100 rows
    if (i + 1) % 100 == 0:
        print(f"✅ Processed {i + 1} rows. Saving...")
        temp_df = pd.DataFrame(gpt_labeled)
        gpt_labeled_df = pd.concat([gpt_labeled_df, temp_df], ignore_index=True)
        gpt_labeled_df.to_csv(checkpoint_file, index=False)
        gpt_labeled = []

# ✅ Final save for remaining rows
if gpt_labeled:
    print("💾 Saving final batch...")
    final_df = pd.DataFrame(gpt_labeled)
    gpt_labeled_df = pd.concat([gpt_labeled_df, final_df], ignore_index=True)
    gpt_labeled_df.to_csv(checkpoint_file, index=False)

print("🏁 GPT labeling finished and saved to:", checkpoint_file)

# ✅ Trigger download
files.download(checkpoint_file)


✅ Processed 100 rows. Saving...


<ipython-input-38-7cb4d9163796>:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gpt_labeled_df = pd.concat([gpt_labeled_df, temp_df], ignore_index=True)


✅ Processed 200 rows. Saving...
✅ Processed 300 rows. Saving...
✅ Processed 400 rows. Saving...
✅ Processed 500 rows. Saving...
✅ Processed 600 rows. Saving...
✅ Processed 700 rows. Saving...
✅ Processed 800 rows. Saving...
✅ Processed 900 rows. Saving...
✅ Processed 1000 rows. Saving...
✅ Processed 1100 rows. Saving...
✅ Processed 1200 rows. Saving...
✅ Processed 1300 rows. Saving...
✅ Processed 1400 rows. Saving...
✅ Processed 1500 rows. Saving...
✅ Processed 1600 rows. Saving...
✅ Processed 1700 rows. Saving...
✅ Processed 1800 rows. Saving...
✅ Processed 1900 rows. Saving...
✅ Processed 2000 rows. Saving...
🏁 GPT labeling finished and saved to: gpt_labeled_checkpoint.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
import pandas as pd

# Load your labeled checkpoint
df = pd.read_csv("gpt_labeled_checkpoint.csv")

# Clean: Set price = NaN where action_type is "No Trade"
df.loc[df["action_type"] == "No Trade", "price"] = pd.NA

# Optional: cast price to float (if needed)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Save cleaned file
cleaned_file = "gpt_labeled_cleaned.csv"
df.to_csv(cleaned_file, index=False)

print(f"✅ Cleaned dataset saved to {cleaned_file}")


✅ Cleaned dataset saved to gpt_labeled_cleaned.csv


### 📄 Step 6.0 — Setup & Load Data

In this step, we begin the model training and auto-filling process.  
We'll:
- Load the `gpt_labeled_cleaned.csv` (our high-quality labeled training set)
- Load the `heards_parsed_phase1.csv` (from regex-based parsing)
- Create a **safe copy** named `heards_final_structured.csv` to work on

This ensures we never overwrite or lose our original files.


In [40]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_squared_error

# Load GPT-labeled cleaned data (for training)
gpt_train = pd.read_csv("gpt_labeled_cleaned.csv").dropna(subset=["action_type"])

# Load regex-parsed CSV
regex_df = pd.read_csv("heards_parsed_phase1.csv")

# Create a working copy for final dataset
final_df = regex_df.copy()
print("✅ Final dataset created — safe copy in memory.")


✅ Final dataset created — safe copy in memory.


### 🛠 Step 6.1 — Preprocess & Vectorize Headlines (Combined GPT + Regex)

In this step, we:
- Load GPT-labeled and regex-parsed datasets
- Combine them to maximize training size
- Vectorize the `headline` column using TF-IDF

We'll use the resulting matrix as input features for all downstream models.


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Load GPT-labeled (high-quality)
gpt_train = pd.read_csv("gpt_labeled_cleaned.csv").dropna(subset=["action_type"])
gpt_train["source"] = "gpt"

# Load regex-parsed data
regex_df = pd.read_csv("heards_parsed_phase1.csv")

# Use regex-filled rows where action_type is NOT null
regex_train = regex_df.dropna(subset=["action_type"])
regex_train = regex_train[["headline", "action_type", "price", "party"]].copy()
regex_train["source"] = "regex"

# Combine both labeled datasets
combined_train = pd.concat([gpt_train, regex_train], ignore_index=True).drop_duplicates(subset=["headline"])

# Print size info
print(f"✅ GPT-labeled rows: {len(gpt_train)}")
print(f"✅ Regex-labeled rows: {len(regex_train)}")
print(f"✅ Combined labeled rows: {len(combined_train)}")

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(combined_train["headline"].astype(str))


✅ GPT-labeled rows: 2000
✅ Regex-labeled rows: 181122
✅ Combined labeled rows: 160376


### 🤖 Step 6.2 — Train Classifier for `action_type`

We train a Random Forest classifier using the combined dataset.

The trained model will be used to fill missing `action_type` values in the final structured dataset.


In [43]:
from sklearn.ensemble import RandomForestClassifier

# ✅ Target variable
y_action = combined_train["action_type"]

# ✅ Train Random Forest classifier
clf_action = RandomForestClassifier(n_estimators=100, random_state=42)
clf_action.fit(X_train, y_action)
print("✅ action_type classifier trained.")

# ✅ Prepare vectorized data for all regex headlines
X_all = vectorizer.transform(regex_df["headline"].astype(str))

# ✅ Work on a safe copy
final_df = regex_df.copy()

# ✅ Boolean mask of missing action_type values
missing_mask = final_df["action_type"].isna()

# ✅ Predict and fill
final_df.loc[missing_mask, "action_type"] = clf_action.predict(X_all[missing_mask.to_numpy()])
final_df.loc[missing_mask, "source"] = "model"

print(f"✅ Filled {missing_mask.sum()} missing action_type rows using model.")


✅ action_type classifier trained.
✅ Filled 34278 missing action_type rows using model.


### 💰 Step 6.3 — Train Regressor for `price`

Now that `action_type` is complete, we move to the `price` field.

We'll:
- Train a `GradientBoostingRegressor` on all rows with known `price` (from GPT + regex)
- Use TF-IDF on the `headline` field as input features
- Predict missing prices in our working copy `final_df`


In [44]:
from sklearn.ensemble import GradientBoostingRegressor

# 🧪 Training data for price
price_train = combined_train.dropna(subset=["price"])
X_price = vectorizer.transform(price_train["headline"].astype(str))
y_price = price_train["price"]

# 🤖 Train the model
reg_price = GradientBoostingRegressor(n_estimators=100, random_state=42)
reg_price.fit(X_price, y_price)
print("✅ price regressor trained.")

# 📈 Predict missing prices in final_df
missing_price = final_df["price"].isna()
X_missing_price = vectorizer.transform(final_df.loc[missing_price, "headline"].astype(str))

final_df.loc[missing_price, "price"] = reg_price.predict(X_missing_price)
final_df.loc[missing_price, "source"] = "model"

print(f"✅ Filled {missing_price.sum()} missing price rows using model.")


✅ price regressor trained.
✅ Filled 61576 missing price rows using model.


### 🧑‍💼 Step 6.4 — Train Classifier for `party`

We'll train a logistic regression model to predict the `party` (trader name) based on the `headline`.

Using:
- Labeled rows from GPT and regex
- TF-IDF features of the `headline`

Then, we’ll fill all missing `party` entries in the final structured dataset.


In [45]:
from sklearn.linear_model import LogisticRegression

# 🧪 Get rows where party is known (from GPT + regex)
party_train = combined_train.dropna(subset=["party"])
X_party = vectorizer.transform(party_train["headline"].astype(str))
y_party = party_train["party"]

# 🤖 Train logistic regression classifier
clf_party = LogisticRegression(max_iter=200, random_state=42)
clf_party.fit(X_party, y_party)
print("✅ party classifier trained.")

# 🧩 Predict missing parties
missing_party = final_df["party"].isna()
X_missing_party = vectorizer.transform(final_df.loc[missing_party, "headline"].astype(str))

final_df.loc[missing_party, "party"] = clf_party.predict(X_missing_party)
final_df.loc[missing_party, "source"] = "model"

print(f"✅ Filled {missing_party.sum()} missing party rows using model.")


✅ party classifier trained.
✅ Filled 65621 missing party rows using model.


In [46]:
# Check % filled per column
filled_summary = (final_df.notna().sum() / len(final_df) * 100).round(1)
filled_summary = filled_summary.sort_values(ascending=False)

print("📊 Final Structured Dataset Completion (% Filled):")
display(filled_summary)


📊 Final Structured Dataset Completion (% Filled):


,0
headline,100.0
id,100.0
updatedDate,100.0
action_type,100.0
party,100.0
source,100.0
price,100.0
commodity,98.3
grade,88.3
location,85.2


In [47]:
# Count unique values in the commodity column
print("🛢️ Unique Commodities:", final_df["commodity"].nunique())
print("\n📦 Commodities Breakdown:")
display(final_df["commodity"].value_counts(dropna=False))


🛢️ Unique Commodities: 1

📦 Commodities Breakdown:


,count
commodity,
HSFO,211635
NaN,3765


### 🧼 Step 7 — Final Data Cleaning & Export

Before dropping unused fields or renaming anything, we first:
- Save the current structured dataset to disk as a versioned backup
- Then proceed to final refinements (drop redundant columns, re-order, rename)

This ensures we can always roll back to this checkpoint.


In [48]:
from google.colab import files

# ✅ Save versioned backup
backup_filename = "heards_structured_cleaned_01.csv"
final_df.to_csv(backup_filename, index=False)
print(f"💾 Backup saved: {backup_filename}")

# ✅ Download to your local machine
files.download(backup_filename)


💾 Backup saved: heards_structured_cleaned_01.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 🧹 Step 7.1 — Drop Redundant Columns

We remove columns that add no analytical or predictive value:
- `commodity`: Only one non-null value ("HSFO") across all rows


In [49]:
# Drop non-informative column
final_df.drop(columns=["commodity"], inplace=True)
print("🧼 Dropped 'commodity' column.")


🧼 Dropped 'commodity' column.


In [50]:
print("🔍 frequency_tag value counts:\n")
print(final_df["frequency_tag"].value_counts(dropna=False))

print("\n📊 volume_min / volume_max sample stats:")
print(final_df[["volume_min", "volume_max"]].describe())


🔍 frequency_tag value counts:

frequency_tag
5 Day        102764
NaN           95081
BalMnth       14917
Any Day        2636
Bal Month         2
Name: count, dtype: int64

📊 volume_min / volume_max sample stats:
          volume_min     volume_max
count  119534.000000  119534.000000
mean       22.979596      22.316462
std         7.046852       7.948603
min        20.000000       2.000000
25%        20.000000      20.000000
50%        20.000000      20.000000
75%        20.000000      20.000000
max        40.000000      40.000000


In [51]:
# Replace typo "Bal Month" → "BalMnth"
final_df["frequency_tag"] = final_df["frequency_tag"].replace("Bal Month", "BalMnth")
print("🔁 Cleaned 'frequency_tag' values.")


🔁 Cleaned 'frequency_tag' values.


### 💾 Step 7.3 — Save Backup Before Dropping Missing Date Rows

We now save a backup of our dataset **before removing rows with missing `start_date` or `end_date`**, in case we want to restore them later.


In [52]:
from google.colab import files

# Save and download backup
backup_filename = "heards_structured_cleaned_02.csv"
final_df.to_csv(backup_filename, index=False)
print(f"💾 Saved backup as: {backup_filename}")

# Trigger download
files.download(backup_filename)


💾 Saved backup as: heards_structured_cleaned_02.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 🧹 Step 7.4 — Drop Rows with Missing Dates

We drop rows where either `start_date` or `end_date` is missing.
These are essential for time-series alignment and chronological modeling.


In [53]:
# Drop rows missing start_date or end_date
before_drop = len(final_df)
final_df = final_df.dropna(subset=["start_date", "end_date"])
after_drop = len(final_df)

print(f"🧹 Dropped {before_drop - after_drop} rows without valid dates.")
print(f"✅ Remaining rows: {after_drop}")


🧹 Dropped 37043 rows without valid dates.
✅ Remaining rows: 178357


In [54]:
# Percentage of missing values per column (after dropping invalid dates)
missing_summary = final_df.isna().mean().round(4) * 100
print("📉 % of Missing Values per Column:\n")
display(missing_summary.sort_values(ascending=False))


📉 % of Missing Values per Column:



,0
frequency_tag,33.34
volume_min,32.98
volume_max,32.98
price_basis,11.23
incoterm,9.82
location,9.23
grade,2.16
headline,0.00
party,0.00
action_type,0.00


### 📦 Step 7.5 — Reorder Columns & Export Final Dataset

Now that our structured data is fully cleaned, we:
- Reorder the columns for readability and usability
- Save and download the final version for modeling or deployment


In [55]:
# Move 'headline' and 'id' to the end for modeling focus
column_order = [
    "updatedDate", "start_date", "end_date",
    "action_type", "party", "price",
    "price_basis", "grade", "incoterm", "location",
    "volume_min", "volume_max", "frequency_tag",
    "source", "headline", "id"
]

# Apply reordering
final_df = final_df[column_order]
print("✅ Columns reordered (headline and ID moved to end).")


✅ Columns reordered (headline and ID moved to end).


In [56]:
# ✅ Save as versioned output
filename = "heards_structured_cleaned_03.csv"
final_df.to_csv(filename, index=False)
print(f"💾 File saved as: {filename}")

# ✅ Download to local machine
from google.colab import files
files.download(filename)


💾 File saved as: heards_structured_cleaned_03.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>